### Map locations offering kapsalons and their average price.

In [25]:
import duckdb
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
from shapely.wkt import loads
from shapely.geometry import Point
import geopandas as gpd

In [26]:
conn=duckdb.connect('my_database.duckdb')

In [ ]:
deliero_kapsalons=conn.execute(
    """ 
    SELECT 
        resto.name, 
        resto.latitude, 
        resto.longitude, 
        resto.postal_code, 
        AVG(CAST(menu.price AS FLOAT)) AS avg_price,
        resto.rating, 
        resto.rating_number, 
FROM deliveroo_restaurants AS resto
JOIN deliveroo_menu_items AS menu
    ON resto.id = menu.restaurant_id
WHERE LOWER(menu.name) LIKE '%kaps%'
GROUP BY 
    resto.name, resto.latitude, resto.longitude, resto.postal_code, resto.rating, 
    resto.rating_number


"""
).fetch_df()
deliero_kapsalons['geometry']=deliero_kapsalons.apply(lambda row: Point(row['longitude'],row['latitude']),axis=1)

deliero_kapsalons["rating_number"] = pd.to_numeric(deliero_kapsalons["rating_number"], errors="coerce")
deliero_kapsalons["rating"] = pd.to_numeric(deliero_kapsalons["rating"], errors="coerce")





deliero_kapsalons['se'] = rating_std / np.sqrt(deliero_kapsalons['rating_number'])

deliero_kapsalons['conf_interval_lower'] = deliero_kapsalons['rating'] - 1.96 * deliero_kapsalons['se']
deliero_kapsalons['conf_interval_upper'] = deliero_kapsalons['rating'] + 1.96 * deliero_kapsalons['se']


deliero_kapsalons['confidence'] = (deliero_kapsalons['conf_interval_lower'] + deliero_kapsalons['conf_interval_upper']) / 2




gdf_kapsalons = gpd.GeoDataFrame(
    deliero_kapsalons,
    geometry='geometry',
    crs="EPSG:4326"
)


belgium = gpd.read_file("https://raw.githubusercontent.com/johan/world.geo.json/master/countries/BEL.geo.json")

gdf_kapsalons.explore(
    column="avg_price",
    cmap="YlOrRd",  
    tooltip=["name", "avg_price", "postal_code", 'confidence','conf_interval_lower','conf_interval_upper'],
    popup=True,
    marker_kwds={"radius": 6},
    legend=True,
    basemap="CartoDB positron"
)

In [ ]:
takeaway_kapsalons=conn.execute(
    """ 
    SELECT 
    resto.name, 
    resto.latitude, 
    resto.longitude, 
    AVG(CAST(menu.price AS FLOAT)) AS avg_price,
    resto.ratings AS rating, 
    resto.ratingsNumber AS rating_number
FROM takeaway_restaurants AS resto
JOIN takeaway_menuItems AS menu
    ON resto.primarySlug = menu.primarySlug
WHERE LOWER(menu.name) LIKE '%kaps%'
GROUP BY 
    resto.name, 
    resto.latitude, 
    resto.longitude, 
    resto.ratings, 
    resto.ratingsNumber



"""
).fetch_df()


takeaway_kapsalons = takeaway_kapsalons[(takeaway_kapsalons['latitude'] != 0) & (takeaway_kapsalons['longitude'] != 0)]

takeaway_kapsalons['geometry']=takeaway_kapsalons.apply(lambda row: Point(row['longitude'],row['latitude']),axis=1)

takeaway_kapsalons["rating_number"] = pd.to_numeric(takeaway_kapsalons["rating_number"], errors="coerce")
takeaway_kapsalons["rating"] = pd.to_numeric(takeaway_kapsalons["rating"], errors="coerce")



rating_std = takeaway_kapsalons['rating'].std()


takeaway_kapsalons['se'] = rating_std / np.sqrt(takeaway_kapsalons['rating_number'])


takeaway_kapsalons['conf_interval_lower'] = takeaway_kapsalons['rating'] - 1.96 * takeaway_kapsalons['se']
takeaway_kapsalons['conf_interval_upper'] = takeaway_kapsalons['rating'] + 1.96 * takeaway_kapsalons['se']


takeaway_kapsalons['confidence'] = (takeaway_kapsalons['conf_interval_lower'] + takeaway_kapsalons['conf_interval_upper']) / 2




gdf_kapsalons = gpd.GeoDataFrame(
    takeaway_kapsalons,
    geometry='geometry',
    crs="EPSG:4326"
)


belgium = gpd.read_file("https://raw.githubusercontent.com/johan/world.geo.json/master/countries/BEL.geo.json")

gdf_kapsalons.explore(
    column="avg_price",
    cmap="YlOrRd",  
    tooltip=["name", "avg_price",  'confidence','conf_interval_lower','conf_interval_upper'],
    popup=True,
    marker_kwds={"radius": 6},
    legend=True,
    basemap="CartoDB positron"
)

In [ ]:
ubereats_kapsalons=conn.execute(
    """ 
    SELECT 
    resto.title as name, 
    resto.location__latitude as latitude, 
    resto.location__longitude as longitude, 
    AVG(CAST(menu.price AS FLOAT)) AS avg_price,
    resto.rating__rating_value AS rating, 
    resto.rating__review_count AS rating_number
FROM ubereats_restaurants AS resto
JOIN ubereats_menu_items AS menu
    ON resto.id = menu.restaurant_id
WHERE LOWER(menu.name) LIKE '%kaps%'
GROUP BY 
    resto.title, 
    resto.location__latitude , 
    resto.location__longitude, 
    resto.rating__rating_value, 
    resto.rating__review_count

"""
).fetch_df()

ubereats_kapsalons['geometry']=ubereats_kapsalons.apply(lambda row: Point(row['longitude'],row['latitude']),axis=1)

ubereats_kapsalons["rating_number"] = pd.to_numeric(ubereats_kapsalons["rating_number"], errors="coerce")
ubereats_kapsalons["rating"] = pd.to_numeric(ubereats_kapsalons["rating"], errors="coerce")



rating_std =ubereats_kapsalons['rating'].std()


ubereats_kapsalons['se'] = rating_std / np.sqrt(ubereats_kapsalons['rating_number'])


ubereats_kapsalons['conf_interval_lower'] = ubereats_kapsalons['rating'] - 1.96 * ubereats_kapsalons['se']
ubereats_kapsalons['conf_interval_upper'] = ubereats_kapsalons['rating'] + 1.96 * ubereats_kapsalons['se']


ubereats_kapsalons['confidence'] = (ubereats_kapsalons['conf_interval_lower'] +ubereats_kapsalons['conf_interval_upper']) / 2




gdf_kapsalons = gpd.GeoDataFrame(
    ubereats_kapsalons,
    geometry='geometry',
    crs="EPSG:4326"
)


belgium = gpd.read_file("https://raw.githubusercontent.com/johan/world.geo.json/master/countries/BEL.geo.json")

gdf_kapsalons.explore(
    column="avg_price",
    cmap="YlOrRd",  
    tooltip=["name", "avg_price",  'confidence','conf_interval_lower','conf_interval_upper'],
    popup=True,
    marker_kwds={"radius": 6},
    legend=True,
    basemap="CartoDB positron"
)

In [ ]:

takeaway_kapsalons = conn.execute(
    """
    SELECT 
        resto.name, 
        resto.latitude, 
        resto.longitude, 
        AVG(CAST(menu.price AS FLOAT)) AS avg_price,
        resto.ratings AS rating, 
        resto.ratingsNumber AS rating_number
    FROM takeaway_restaurants AS resto
    JOIN takeaway_menuItems AS menu
        ON resto.primarySlug = menu.primarySlug
    WHERE LOWER(menu.name) LIKE '%kaps%'
    GROUP BY 
        resto.name, 
        resto.latitude, 
        resto.longitude, 
        resto.ratings, 
        resto.ratingsNumber
    """
).fetch_df()


takeaway_kapsalons['rating_number'] = pd.to_numeric(takeaway_kapsalons['rating_number'], errors='coerce')
takeaway_kapsalons['rating'] = pd.to_numeric(takeaway_kapsalons['rating'], errors='coerce')


takeaway_kapsalons['rating_number'].fillna(0, inplace=True)
takeaway_kapsalons['rating'].fillna(0, inplace=True)


rating_std = takeaway_kapsalons['rating'].std()


takeaway_kapsalons['se'] = np.where(
    takeaway_kapsalons['rating_number'] > 0,
    rating_std / np.sqrt(takeaway_kapsalons['rating_number']),
    np.nan  
)


takeaway_kapsalons['conf_interval_lower'] = np.where(
    takeaway_kapsalons['se'].notna(),
    takeaway_kapsalons['rating'] - 1.96 * takeaway_kapsalons['se'],
    np.nan
)
takeaway_kapsalons['conf_interval_upper'] = np.where(
    takeaway_kapsalons['se'].notna(),
    takeaway_kapsalons['rating'] + 1.96 * takeaway_kapsalons['se'],
    np.nan
)


takeaway_kapsalons['conf_interval_lower'] = takeaway_kapsalons['conf_interval_lower'].replace([np.inf, -np.inf], np.nan)
takeaway_kapsalons['conf_interval_upper'] = takeaway_kapsalons['conf_interval_upper'].replace([np.inf, -np.inf], np.nan)


takeaway_kapsalons['confidence'] = (takeaway_kapsalons['conf_interval_lower'] + takeaway_kapsalons['conf_interval_upper']) / 2

takeaway_kapsalons['geometry']=takeaway_kapsalons.apply(lambda row: Point(row['longitude'],row['latitude']),axis=1)


gdf_kapsalons = gpd.GeoDataFrame(
    takeaway_kapsalons,
    geometry='geometry',
    crs="EPSG:4326"
)


belgium = gpd.read_file("https://raw.githubusercontent.com/johan/world.geo.json/master/countries/BEL.geo.json")

gdf_kapsalons.explore(
    column="avg_price",
    cmap="YlOrRd",  
    tooltip=["name", "avg_price",  'confidence','conf_interval_lower','conf_interval_upper'],
    popup=True,
    marker_kwds={"radius": 6},
    legend=True,
    basemap="CartoDB positron"
)


C:\Users\ilasv\AppData\Local\Temp\ipykernel_18612\1981902012.py:29: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  takeaway_kapsalons['rating_number'].fillna(0, inplace=True)
C:\Users\ilasv\AppData\Local\Temp\ipykernel_18612\1981902012.py:30: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a c